# Bayesian Optimisation — Function 4 (4D)

**Author:** Dibyajyoti Pradhan  
**Programme:** Imperial College London Professional Certificate in ML/AI  
**Module:** BBO Capstone (Modules 12–22)  

**Strategy note:** Converging cluster with consistent gains. Tight local refinement.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add src/ to path for reusable utilities
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from bbo_utils import (
    lhs_candidates, fit_gp, gp_posterior,
    ucb_acquisition, suggest_next_query,
    dimension_sensitivity, plot_running_best,
)

np.random.seed(42)

## 2. Load Initial Data

Initial observations provided at the start of the BBO challenge.

In [ ]:
data_dir = os.path.join('..', 'data', 'initial_data', 'function_4')

X_init = np.load(os.path.join(data_dir, 'initial_inputs.npy'))
y_init = np.load(os.path.join(data_dir, 'initial_outputs.npy'))

print(f'Initial inputs shape:  {X_init.shape}')
print(f'Initial outputs shape: {y_init.shape}')
print(f'Output range: [{y_init.min():.6f}, {y_init.max():.6f}]')
print(f'Best initial output:   {y_init.max():.6f}')
print(f'Best initial input:    {X_init[np.argmax(y_init)]}')

## 3. Add Subsequent Query Data

Queries from Weeks 1–5 with confirmed outputs.  
Week 6 submitted — awaiting results (not included in training).

In [ ]:
# Weekly queries — Weeks 1–5 (confirmed outputs)
queries_X = np.array([
    [0.009153, 0.333687, 0.980781, 0.997681],   # W1
    [0.514874, 0.428474, 0.422867, 0.202058],   # W2
    [0.534616, 0.470071, 0.430432, 0.237463],   # W3
    [0.587976, 0.452666, 0.456567, 0.174989],   # W4
    [0.489738, 0.422114, 0.504747, 0.243196],   # W5
])
queries_y = np.array([
    -34.09002216462411,    # W1
     -3.796547971527221,   # W2
     -3.769558238448095,   # W3
     -6.225383151225145,   # W4
     -3.424938968473334,   # W5
])

# Week 6 submitted — awaiting result (excluded from training)
# x_w6 = np.array([0.528154, 0.438581, 0.469133, 0.231649])

X_train = np.vstack([X_init, queries_X])
y_train = np.append(y_init, queries_y)

print(f'Total observations: {len(y_train)}')
print(f'Best observed output: {y_train.max():.6f}')
print(f'Best observed input:  {X_train[np.argmax(y_train)]}')

## 4. Data Exploration

In [ ]:
print('Output statistics:')
print(f'  Min:  {y_train.min():.6f}')
print(f'  Max:  {y_train.max():.6f}')
print(f'  Mean: {y_train.mean():.6f}')
print(f'  Std:  {y_train.std():.6f}')

# Running best across all observations
plot_running_best(y_train, title='Function 4 — Running Best Output')

### Dimension–Output Correlation

For high-dimensional functions, identifying influential dimensions allows pseudo-dimensionality reduction (fixing low-influence inputs).

In [ ]:
correlations = np.array([
    np.corrcoef(X_train[:, i], y_train)[0, 1]
    for i in range(X_train.shape[1])
])

plt.figure(figsize=(8, 4))
plt.bar([f'X{i+1}' for i in range(4)], correlations, color='steelblue')
plt.axhline(0, color='k', linewidth=0.8)
plt.ylabel('Pearson r with output')
plt.title('Function 4 — Dimension-Output Correlation')
plt.grid(axis='y'); plt.tight_layout(); plt.show()

print('Correlations (X1, X2, X3, X4):')
for i, r in enumerate(correlations):
    print(f'  X{i+1}: {r:.4f}')

## 5. Bayesian Optimisation (UCB, β=0.2)

Strategy: exploitation-heavy (trust region ±0.05)

In [ ]:
next_query, fitted_model = suggest_next_query(
    X_train=X_train,
    y_train=y_train,
    n_candidates=2_000_000,
    beta=0.2,
    length_scale=0.1,
    noise_level=1e-6,
    fit_noise=True,
    trust_region_radius=0.05,
    seed=42,
)

# Format to 6 decimal places (BBO portal requirement)
coords = ', '.join(f'{v:.6f}' for v in next_query)
print(f'Week 7 suggested query for Function 4:')
print(f'  ({coords})')

### Dimension Sensitivity (Finite-Difference Gradients)

Identifies which input dimensions most strongly influence the GP predicted mean at the current best point.

In [ ]:
best_x = X_train[np.argmax(y_train)]
sensitivity = dimension_sensitivity(fitted_model, best_x, delta=0.01)

plt.figure(figsize=(8, 4))
plt.bar([f'X{i+1}' for i in range(4)], sensitivity, color='tomato')
plt.axhline(0, color='k', linewidth=0.8)
plt.ylabel('∂μ/∂xᵢ  (GP mean gradient)')
plt.title('Function 4 — Dimension Sensitivity at Best Point')
plt.grid(axis='y'); plt.tight_layout(); plt.show()

print('Sensitivity at best point:')
for i, g in enumerate(sensitivity):
    print(f'  X{i+1}: {g:+.4f}')

## 6. Week 7 Strategy Summary

| Field | Value |
|-------|-------|
| Function | F4 (4D) |
| Total observations | n/a (computed above) |
| Acquisition | UCB, β=0.2 |
| Trust region | ±0.05 per dim |
| Strategy | Best at W5 (-3.425). Refine near [0.49, 0.42, 0.50, 0.24]. |

> Submit the coordinates printed above via the BBO portal. All inputs must be in `[0, 1)` to 6 decimal places.